In [1]:
!pip install -q "scrapling[all]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.5/116.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.2/624.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 726.5/726.5 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.11.0 which is incompatible.
moviepy 1.0.

In [3]:
from scrapling.cli import install
install([], standalone_mode=False)

The dependencies are already installed


In [ ]:
import os
import requests
import urllib.parse
import asyncio
from concurrent.futures import ThreadPoolExecutor
from scrapling.fetchers import AsyncStealthySession

output_dir = "massive_arabic_menus"
os.makedirs(output_dir, exist_ok=True)

# 1. Base Keywords
base_keywords = [
    "منيو مطعم مصري", "منيوهات مطاعم", "منيو كشري", "اسعار منيو مشويات",
    "منيو شاورما", "منيو بيتزا مصري", "منيو كافيه", "قائمة اسعار مطعم",
    "منيو اسماك", "منيو فول وفلافل", "منيو برجر", "منيو كبدة ومخ",
    "منيو حواوشي", "منيو فطاير", "منيو حلويات مصري", "منيو بروست",
    "منيو مطاعم اسكندرية", "منيو مطاعم القاهرة", "قائمة طعام كافيه",
    "منيو وجبات سريعة", "منيو فطار مصري", "منيو menuegypt", "منيو مطعم", "منيو فاست فود"
]

locations = [
    "", " القاهرة", " الجيزة", " الاسكندرية", " المنصورة",
    " طنطا", " الزقازيق", " اسيوط", " التجمع", " مدينة نصر"
]

expanded_keywords = []
for base in base_keywords:
    for loc in locations:
        expanded_keywords.append(base + loc)

print(f"Generated {len(expanded_keywords)} unique search queries.")

def download_image(url, kw_idx, img_idx):
    if not url.startswith("http"):
        return
    try:
        res = requests.get(url, timeout=10)
        if res.status_code == 200:
            ext = url.split('.')[-1].split('?')[0]
            if ext.lower() not in ['jpg', 'jpeg', 'png', 'webp']:
                ext = 'jpg'
            filename = os.path.join(output_dir, f"menu_{kw_idx}_{img_idx}.{ext}")
            with open(filename, "wb") as f:
                f.write(res.content)
    except:
        pass

all_image_urls = []

# --- THE ASYNC FIX ---
async def harvest_urls():
    print("Booting up Scrapling ASYNC Stealth Fetcher...")

    # We open ONE persistent, stealthy asynchronous browser session
    async with AsyncStealthySession(headless=True) as session:

        for kw_idx, keyword in enumerate(expanded_keywords[:100]):
            print(f"Scraping Bing safely for: {keyword}")
            encoded_kw = urllib.parse.quote(keyword)
            search_url = f"https://www.bing.com/images/search?q={encoded_kw}&form=HDRSC2"

            try:
                # We use 'await' to let Colab breathe while it fetches the page
                page = await session.fetch(search_url, timeout=60000)
                image_elements = page.css('a.iusc')

                found_count = 0
                for img_idx, element in enumerate(image_elements):
                    m_data = element.attrib.get('m')
                    if m_data and '"murl":"' in m_data:
                        start = m_data.find('"murl":"') + 8
                        end = m_data.find('"', start)
                        img_url = m_data[start:end]
                        all_image_urls.append((img_url, kw_idx, img_idx))
                        found_count += 1

                print(f" -> Successfully found {found_count} links.")

            except Exception as e:
                print(f" -> Failed on {keyword}: {str(e)}")

# Execute the async function natively in Colab
await harvest_urls()

print(f"\nScrapling successfully bypassed security and found {len(all_image_urls)} high-res image links!")
print("Downloading to Colab disk...")

# Download all extracted URLs using multiple threads to speed it up
with ThreadPoolExecutor(max_workers=15) as executor:
    for url, kw_idx, img_idx in all_image_urls:
        executor.submit(download_image, url, kw_idx, img_idx)

final_count = len(os.listdir(output_dir))
print(f"\n✅ Total valid images securely downloaded: {final_count}")

In [7]:
final_count = len(os.listdir(output_dir))
print(f"\n✅ Total valid images securely downloaded: {final_count}")


✅ Total valid images securely downloaded: 3344


In [8]:
from google.colab import files

print("Compressing 3,300+ images into a single ZIP file...")
# This creates a file named 'massive_arabic_menus.zip'
!zip -r -q massive_arabic_menus.zip massive_arabic_menus/

print("Compression complete! Triggering browser download...")
# This will prompt your browser to save the file to your PC
files.download('massive_arabic_menus.zip')

Compressing 3,300+ images into a single ZIP file...
Compression complete! Triggering browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>